In [ ]:
from pathlib import Path
from zipfile import ZipFile
from collections import Counter

# Your ZIP file
ZIP_PATH = Path(r"112x112 main dataset.zip")

# Extract to a separate folder, leaving the ZIP unchanged
EXTRACT_DIR = Path(
    r"curricularface_run"
)

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
EXPECTED_CLASSES = {"member_1", "member_2", "member_3", "member_4"}

NAME_BY_FOLDER = {
    "member_1": "Team Member 1",
    "member_2": "Team Member 2",
    "member_3": "Team Member 3",
    "member_4": "Team Member 4",
}

if not ZIP_PATH.is_file():
    raise FileNotFoundError(
        f"ZIP not found at:\n{ZIP_PATH}\n\n"
        "Check that you downloaded the ZIP to this location "
        "and that its filename matches exactly."
    )

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with ZipFile(ZIP_PATH, "r") as archive:
    bad_file = archive.testzip()
    if bad_file is not None:
        raise RuntimeError(f"The ZIP contains a damaged file: {bad_file}")

    archive.extractall(EXTRACT_DIR)

# Find the folder that directly contains train, validate, and test
train_folders = [
    path for path in EXTRACT_DIR.rglob("train")
    if path.is_dir()
    and (path.parent / "validate").is_dir()
    and (path.parent / "test").is_dir()
]

if len(train_folders) != 1:
    raise RuntimeError(
        "Could not identify one unique dataset folder containing "
        "train, validate, and test. Folders found:\n"
        + "\n".join(str(path) for path in train_folders)
    )

DATA_ROOT = train_folders[0].parent

# Check the four identity folders and count images in each split
for split in ("train", "validate", "test"):
    split_dir = DATA_ROOT / split
    class_folders = {
        path.name for path in split_dir.iterdir()
        if path.is_dir()
    }

    print(f"\n{split.upper()} folder: {split_dir}")
    print("Class folders:", sorted(class_folders))

    if class_folders != EXPECTED_CLASSES:
        raise RuntimeError(
            f"Unexpected folders in {split}: {sorted(class_folders)}"
        )

    for class_name in sorted(class_folders):
        count = sum(
            1 for path in (split_dir / class_name).rglob("*")
            if path.is_file()
            and path.suffix.lower() in IMAGE_EXTENSIONS
        )
        print(f"  {class_name} ({NAME_BY_FOLDER[class_name]}): {count}")

print("\nDATA_ROOT =", DATA_ROOT)
print("Dataset extraction and folder check complete.")

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# CurricularFace input dimensions
IMAGE_SIZE = 112

# Use the GPU if PyTorch can access one; otherwise use the CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Training-only light augmentation. Validation and test remain unchanged.
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(
        brightness=0.10,
        contrast=0.10
    ),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

train_dataset = datasets.ImageFolder(
    DATA_ROOT / "train",
    transform=train_transform
)

validate_dataset = datasets.ImageFolder(
    DATA_ROOT / "validate",
    transform=eval_transform
)

test_dataset = datasets.ImageFolder(
    DATA_ROOT / "test",
    transform=eval_transform
)

# Keep class indices consistent across all three splits
class_names = train_dataset.classes

assert set(class_names) == set(NAME_BY_FOLDER), (
    f"Unexpected class folders: {class_names}"
)
assert validate_dataset.class_to_idx == train_dataset.class_to_idx
assert test_dataset.class_to_idx == train_dataset.class_to_idx

class_to_idx = train_dataset.class_to_idx
idx_to_name = {
    index: NAME_BY_FOLDER[folder]
    for folder, index in class_to_idx.items()
}
num_classes = len(class_names)

# Small batches and num_workers=0 are safer on a Windows laptop
BATCH_SIZE = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda")
)

validate_loader = DataLoader(
    validate_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda")
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda")
)

# Check one batch and show the label mapping
images, labels = next(iter(train_loader))

print("Device:", DEVICE)
print("DATA_ROOT:", DATA_ROOT)
print("Folders:", class_names)
print("Display names:", idx_to_name)
print("Train files:", len(train_dataset))
print("Validation files:", len(validate_dataset))
print("Test files:", len(test_dataset))
print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)

In [ ]:
# Complete CurricularFace setup: run this cell after restarting the kernel.

from pathlib import Path
from zipfile import ZipFile
from collections import Counter
import importlib
import math
import random
import sys
import types

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms


# ------------------------------------------------------------
# 1. Reproducibility and device
# ------------------------------------------------------------
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMAGE_SIZE = 112
BATCH_SIZE = 8


# ------------------------------------------------------------
# 2. Extract the dataset ZIP and locate train/validate/test
# ------------------------------------------------------------
ZIP_PATH = Path(
    r"112x112 main dataset.zip"
)

if not ZIP_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset ZIP not found:\n{ZIP_PATH}\n"
        "Check the filename and location."
    )

# Use a fresh extraction folder so older files cannot remain mixed in.
EXTRACT_BASE = ZIP_PATH.parent / "curricularface_extracted"
EXTRACT_DIR = EXTRACT_BASE
suffix_number = 2

while EXTRACT_DIR.exists():
    EXTRACT_DIR = ZIP_PATH.parent / f"curricularface_extracted_{suffix_number}"
    suffix_number += 1

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with ZipFile(ZIP_PATH, "r") as archive:
    damaged_file = archive.testzip()
    if damaged_file is not None:
        raise RuntimeError(f"Damaged file inside ZIP: {damaged_file}")
    archive.extractall(EXTRACT_DIR)

train_folders = [
    path for path in EXTRACT_DIR.rglob("train")
    if path.is_dir()
    and (path.parent / "validate").is_dir()
    and (path.parent / "test").is_dir()
]

if len(train_folders) != 1:
    raise RuntimeError(
        "Could not find one dataset folder containing train, validate, and test.\n"
        f"Found: {[str(path) for path in train_folders]}"
    )

DATA_ROOT = train_folders[0].parent


# ------------------------------------------------------------
# 3. Identity labels and display names
# ------------------------------------------------------------
FOLDER_TO_NAME = {
    "member_1": "Team Member 1",
    "member_2": "Team Member 2",
    "member_3": "Team Member 3",
    "member_4": "Team Member 4",
}

class_names = sorted(
    folder.name
    for folder in (DATA_ROOT / "train").iterdir()
    if folder.is_dir()
)

if set(class_names) != set(FOLDER_TO_NAME):
    raise RuntimeError(f"Unexpected identity folders: {class_names}")

class_to_idx = {name: index for index, name in enumerate(class_names)}
idx_to_name = {
    class_to_idx[folder]: FOLDER_TO_NAME[folder]
    for folder in class_names
}

num_classes = len(class_names)
NUM_CLASSES = num_classes


# ------------------------------------------------------------
# 4. Image transforms and data loaders
# ------------------------------------------------------------
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.10, contrast=0.10),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

train_dataset = datasets.ImageFolder(
    DATA_ROOT / "train",
    transform=train_transform
)
validate_dataset = datasets.ImageFolder(
    DATA_ROOT / "validate",
    transform=eval_transform
)
test_dataset = datasets.ImageFolder(
    DATA_ROOT / "test",
    transform=eval_transform
)

if not (
    train_dataset.class_to_idx
    == validate_dataset.class_to_idx
    == test_dataset.class_to_idx
):
    raise RuntimeError("Class-to-index mapping differs between dataset splits.")

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda")
)
validate_loader = DataLoader(
    validate_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda")
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda")
)


# ------------------------------------------------------------
# 5. CurricularFace classification head
# ------------------------------------------------------------
class CurricularFaceHead(nn.Module):
    def __init__(
        self,
        embedding_dim,
        num_classes,
        scale=64.0,
        margin=0.5,
        momentum=0.99
    ):
        super().__init__()

        self.scale = scale
        self.momentum = momentum

        self.weight = nn.Parameter(
            torch.empty(num_classes, embedding_dim)
        )
        nn.init.normal_(self.weight, std=0.01)

        self.cos_m = math.cos(margin)
        self.sin_m = math.sin(margin)
        self.threshold = math.cos(math.pi - margin)
        self.mm = math.sin(math.pi - margin) * margin

        self.register_buffer("t", torch.zeros(1))

    def forward(self, embeddings, labels=None):
        embeddings = F.normalize(embeddings, dim=1)
        normalized_weights = F.normalize(self.weight, dim=1)

        cosine = F.linear(embeddings, normalized_weights)
        cosine = cosine.clamp(-1.0 + 1e-7, 1.0 - 1e-7)

        # Inference: return cosine scores without applying the training margin.
        if labels is None:
            return cosine * self.scale

        labels = labels.long()
        target_cosine = cosine.gather(1, labels.view(-1, 1))

        sine = torch.sqrt(
            (1.0 - target_cosine.square()).clamp_min(1e-7)
        )
        target_with_margin = (
            target_cosine * self.cos_m
            - sine * self.sin_m
        )

        target_logit = torch.where(
            target_cosine > self.threshold,
            target_with_margin,
            target_cosine - self.mm
        )

        with torch.no_grad():
            self.t.mul_(self.momentum).add_(
                target_cosine.mean() * (1.0 - self.momentum)
            )

        hard_mask = cosine > target_with_margin
        adjusted_cosine = torch.where(
            hard_mask,
            cosine * (self.t + cosine),
            cosine
        )

        logits = adjusted_cosine.scatter(
            1,
            labels.view(-1, 1),
            target_logit
        )

        return logits * self.scale


# ------------------------------------------------------------
# 6. Load the face backbone and build the model
# ------------------------------------------------------------
# Compatibility alias for an old checkpoint import path.
try:
    importlib.import_module("torch.utils.serialization")
except ModuleNotFoundError as error:
    if error.name != "torch.utils.serialization":
        raise

    import torch.utils

    legacy_serialization = types.ModuleType("torch.utils.serialization")
    legacy_serialization.__dict__.update(torch.serialization.__dict__)
    sys.modules["torch.utils.serialization"] = legacy_serialization
    torch.utils.serialization = legacy_serialization

from facenet_pytorch import InceptionResnetV1


class CurricularFaceModel(nn.Module):
    def __init__(self, class_count):
        super().__init__()

        self.backbone = InceptionResnetV1(
            pretrained="vggface2",
            classify=False
        )
        self.head = CurricularFaceHead(
            embedding_dim=512,
            num_classes=class_count
        )

    def forward(self, images, labels=None):
        embeddings = self.backbone(images)
        return self.head(embeddings, labels)


model = CurricularFaceModel(num_classes).to(DEVICE)

# Freeze most backbone weights; fine-tune the last layers and the head.
for parameter in model.backbone.parameters():
    parameter.requires_grad = False

for layer_name in ("repeat_3", "block8", "last_linear", "last_bn"):
    layer = getattr(model.backbone, layer_name, None)
    if layer is not None:
        for parameter in layer.parameters():
            parameter.requires_grad = True


# ------------------------------------------------------------
# 7. Training setup and a small model-output check
# ------------------------------------------------------------
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    (parameter for parameter in model.parameters()
     if parameter.requires_grad),
    lr=1e-4,
    weight_decay=1e-4
)

OUTPUT_DIR = ZIP_PATH.parent / "curricularface_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NUM_EPOCHS = 8
best_val_accuracy = 0.0

sample_images, sample_labels = next(iter(train_loader))
model.eval()

with torch.no_grad():
    sample_scores = model(sample_images[:1].to(DEVICE))

assert sample_scores.shape == (1, num_classes), (
    f"Expected one score per class; got {sample_scores.shape}"
)

model.train()

print("Setup complete.")
print("Dataset folder:", DATA_ROOT)
print("Device:", DEVICE)
print("Classes:", idx_to_name)
print(
    "Images — train:", len(train_dataset),
    "| validate:", len(validate_dataset),
    "| test:", len(test_dataset)
)
print("Model output shape:", sample_scores.shape)
print(
    "Trainable parameters:",
    f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}"
)
print("Results folder:", OUTPUT_DIR)

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class CurricularFaceHead(nn.Module):
    def __init__(
        self,
        embedding_dim,
        num_classes,
        scale=64.0,
        margin=0.5,
        momentum=0.99
    ):
        super().__init__()

        self.embedding_dim = embedding_dim
        self.num_classes = num_classes
        self.scale = scale
        self.margin = margin
        self.momentum = momentum

        # One class weight vector per identity
        self.weight = nn.Parameter(
            torch.empty(num_classes, embedding_dim)
        )
        nn.init.normal_(self.weight, std=0.01)

        self.cos_m = math.cos(margin)
        self.sin_m = math.sin(margin)
        self.threshold = math.cos(math.pi - margin)
        self.mm = math.sin(math.pi - margin) * margin

        # CurricularFace's running estimate of positive cosine similarity
        self.register_buffer("t", torch.zeros(1))

    def forward(self, embeddings, labels=None):
        embeddings = F.normalize(embeddings, dim=1)
        class_weights = F.normalize(self.weight, dim=1)

        cosine = F.linear(embeddings, class_weights)
        cosine = cosine.clamp(-1.0 + 1e-7, 1.0 - 1e-7)

        # During validation, testing, and webcam inference, use cosine scores.
        if labels is None:
            return cosine * self.scale

        labels = labels.long()
        target_cosine = cosine.gather(1, labels.view(-1, 1))

        sine = torch.sqrt(
            (1.0 - target_cosine.square()).clamp_min(1e-7)
        )
        target_with_margin = (
            target_cosine * self.cos_m
            - sine * self.sin_m
        )

        target_logit = torch.where(
            target_cosine > self.threshold,
            target_with_margin,
            target_cosine - self.mm
        )

        # Update the curriculum state without tracking gradients
        with torch.no_grad():
            batch_mean = target_cosine.mean()
            self.t.mul_(self.momentum).add_(
                batch_mean * (1.0 - self.momentum)
            )

        # Adjust hard negative classes using the running curriculum value
        hard_mask = cosine > target_with_margin
        adjusted_cosine = torch.where(
            hard_mask,
            cosine * (self.t + cosine),
            cosine
        )

        logits = adjusted_cosine.scatter(
            1,
            labels.view(-1, 1),
            target_logit
        )

        return logits * self.scale


print("CurricularFace head defined.")
print("Number of identity classes:", num_classes)

In [ ]:
import importlib
import sys
import types
import torch
import torch.nn as nn

# Compatibility alias for older cached PyTorch checkpoints that reference
# the removed module path torch.utils.serialization.
try:
    importlib.import_module("torch.utils.serialization")
except ModuleNotFoundError:
    legacy_serialization = types.ModuleType("torch.utils.serialization")
    legacy_serialization.__dict__.update(torch.serialization.__dict__)
    sys.modules["torch.utils.serialization"] = legacy_serialization
    torch.utils.serialization = legacy_serialization

from facenet_pytorch import InceptionResnetV1


class CurricularFaceModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        # 512-dimensional face embeddings, pretrained on VGGFace2
        self.backbone = InceptionResnetV1(
            pretrained="vggface2",
            classify=False
        )

        self.head = CurricularFaceHead(
            embedding_dim=512,
            num_classes=num_classes,
            scale=64.0,
            margin=0.5,
            momentum=0.99
        )

    def forward(self, images, labels=None):
        embeddings = self.backbone(images)
        return self.head(embeddings, labels)


model = CurricularFaceModel(num_classes).to(DEVICE)

# Freeze most of the pretrained backbone to reduce CPU training time
for parameter in model.backbone.parameters():
    parameter.requires_grad = False

# Fine-tune only the last backbone blocks and the CurricularFace head
for layer_name in ("repeat_3", "block8", "last_linear", "last_bn"):
    layer = getattr(model.backbone, layer_name, None)
    if layer is not None:
        for parameter in layer.parameters():
            parameter.requires_grad = True

# Check that a batch produces four class scores
model.eval()
with torch.no_grad():
    sample_scores = model(images[:2].to(DEVICE))

assert sample_scores.shape == (2, num_classes), (
    f"Unexpected output shape: {sample_scores.shape}"
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

model.train()

print("Model ready on:", DEVICE)
print("Output shape:", sample_scores.shape)
print("Trainable parameters:", f"{trainable_parameters:,}")
print("Class order:", idx_to_name)

In [ ]:
from pathlib import Path
import torch
import torch.nn as nn

# Training settings
EPOCHS = 8
criterion = nn.CrossEntropyLoss()

# Optimize the CurricularFace head and the backbone layers you unfroze
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-4,
    weight_decay=1e-4
)

# Save results on your Desktop
OUTPUT_DIR = Path(
    r"curricularface_results"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = OUTPUT_DIR / "curricularface_best.pt"
class_names = train_loader.dataset.classes
history = []
best_val_accuracy = -1.0


def run_one_epoch(loader, training):
    if training:
        model.train()

        # Keep most pretrained backbone batch-normalization layers fixed
        model.backbone.eval()

        # Put only the unfrozen final layers into training mode
        for layer_name in ("repeat_3", "block8", "last_linear", "last_bn"):
            layer = getattr(model.backbone, layer_name, None)
            if layer is not None and any(
                parameter.requires_grad for parameter in layer.parameters()
            ):
                layer.train()

        model.head.train()
    else:
        model.eval()

    total_loss = 0.0
    total_correct = 0
    total_images = 0

    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        if training:
            optimizer.zero_grad(set_to_none=True)

            # Labels are supplied during training so the CurricularFace
            # margin and curriculum update are applied.
            logits = model(images, labels)
            loss = criterion(logits, labels)

            loss.backward()
            optimizer.step()
        else:
            # No labels here: validation uses plain class cosine scores.
            with torch.no_grad():
                logits = model(images)
                loss = criterion(logits, labels)

        batch_size = images.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_images += batch_size

    average_loss = total_loss / total_images
    accuracy = total_correct / total_images
    return average_loss, accuracy


for epoch in range(1, EPOCHS + 1):
    train_loss, train_accuracy = run_one_epoch(train_loader, training=True)
    val_loss, val_accuracy = run_one_epoch(validate_loader, training=False)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
    })

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"train loss {train_loss:.4f} | "
        f"train accuracy {train_accuracy:.1%} | "
        f"validation loss {val_loss:.4f} | "
        f"validation accuracy {val_accuracy:.1%}"
    )

    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "class_names": class_names,
                "best_validation_accuracy": best_val_accuracy,
                "epoch": epoch,
            },
            CHECKPOINT_PATH
        )
        print("Saved best checkpoint:", CHECKPOINT_PATH)

print(f"\nBest validation accuracy: {best_val_accuracy:.1%}")
print("Best model saved to:", CHECKPOINT_PATH)

In [ ]:
from pathlib import Path
from collections import Counter
import hashlib

import torch
from torch.utils.data import DataLoader, Subset


CHECKPOINT_PATH = Path(
    r"curricularface_results/curricularface_best.pt"
)

# Load the checkpoint with the best validation score
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=True
)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

class_names = test_dataset.classes

if "idx_to_name" in globals():
    display_names = idx_to_name
else:
    display_names = {
        index: name for index, name in enumerate(class_names)
    }


# Build a test subset with one copy of each exact image per identity
seen_hashes = {}
unique_indices = []
duplicates_skipped = 0

for index, (image_path, label) in enumerate(test_dataset.samples):
    digest = hashlib.sha256(Path(image_path).read_bytes()).hexdigest()

    if digest in seen_hashes:
        previous_label = seen_hashes[digest]

        if previous_label != label:
            raise RuntimeError(
                "The same exact test image has conflicting identity labels:\n"
                f"{image_path}"
            )

        duplicates_skipped += 1
        continue

    seen_hashes[digest] = label
    unique_indices.append(index)

unique_test_loader = DataLoader(
    Subset(test_dataset, unique_indices),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)


def evaluate_test_loader(loader):
    confusion = torch.zeros(
        len(class_names),
        len(class_names),
        dtype=torch.int64
    )
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            # No labels passed here: inference uses cosine scores.
            scores = model(images)
            predictions = scores.argmax(dim=1)

            for actual, predicted in zip(labels.cpu(), predictions.cpu()):
                confusion[int(actual), int(predicted)] += 1

            total += labels.size(0)

    correct = int(confusion.diag().sum())
    accuracy = correct / total if total else 0.0
    return accuracy, confusion, total


file_accuracy, file_confusion, file_total = evaluate_test_loader(test_loader)
unique_accuracy, unique_confusion, unique_total = evaluate_test_loader(
    unique_test_loader
)

print(f"Best checkpoint epoch: {checkpoint['epoch']}")
print(f"Best validation accuracy: {checkpoint['best_validation_accuracy']:.1%}")

print(f"\nAll test files: {file_total}")
print(f"All-file test accuracy: {file_accuracy:.1%}")

print(f"\nUnique test images: {unique_total}")
print(f"Exact duplicate files skipped: {duplicates_skipped}")
print(f"Unique-image test accuracy: {unique_accuracy:.1%}")

print("\nPer-identity accuracy on unique test images:")
for index, folder_name in enumerate(class_names):
    count = int(unique_confusion[index].sum())
    correct = int(unique_confusion[index, index])
    accuracy = correct / count if count else 0.0
    print(f"{display_names[index]}: {correct}/{count} ({accuracy:.1%})")

print("\nConfusion matrix for unique test images")
print("Rows = actual identity; columns = predicted identity")
print("Class order:", [display_names[i] for i in range(len(class_names))])
print(unique_confusion)

In [ ]:
# CurricularFace: calibrate the unknown-person rejection threshold
# Run after the cells that define the model and validation dataset.

from pathlib import Path
from PIL import Image
from sklearn.datasets import fetch_lfw_people
import hashlib
import json
import random
import numpy as np
import torch

# ---------- Find the validation dataset ----------
validation_data = None
for variable_name in ("validate_dataset", "val_dataset", "val_ds", "valid_dataset"):
    if variable_name in globals():
        validation_data = globals()[variable_name]
        break

if validation_data is None:
    raise NameError(
        "Validation dataset not found. Run the dataset setup cell first."
    )

if "model" not in globals():
    raise NameError("CurricularFace model not found. Run the model setup cell first.")

# ---------- Paths and settings ----------
RESULTS_DIR = Path(
    r"curricularface_results"
)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = RESULTS_DIR / "curricularface_best.pt"
SETTINGS_PATH = RESULTS_DIR / "curricularface_settings.json"
LFW_CACHE_DIR = RESULTS_DIR / "lfw_cache"

# Use the notebook's evaluation transform if it exists.
if "eval_transform" in globals():
    evaluation_transform = eval_transform
else:
    from torchvision import transforms

    evaluation_transform = transforms.Compose([
        transforms.Resize((112, 112)),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ])

# Use the already-defined team names to exclude them from the unknown set.
folder_to_name = {}
for mapping_name in ("FOLDER_TO_NAME", "NAME_BY_FOLDER", "ID_TO_NAME"):
    mapping = globals().get(mapping_name)
    if isinstance(mapping, dict):
        folder_to_name = mapping
        break

# The ImageFolder dataset provides its class folder names and labels.
class_folders = list(validation_data.classes)
class_to_idx = validation_data.class_to_idx

idx_to_name = {
    int(class_to_idx[folder]): str(folder_to_name.get(folder, folder))
    for folder in class_folders
}

# ---------- Load the best CurricularFace checkpoint ----------
device = next(model.parameters()).device

if CHECKPOINT_PATH.is_file():
    try:
        checkpoint = torch.load(
            CHECKPOINT_PATH,
            map_location=device,
            weights_only=True
        )
    except TypeError:
        # Compatibility with older PyTorch versions.
        checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)

    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
    else:
        model.load_state_dict(checkpoint)

    print("Loaded best CurricularFace checkpoint:", CHECKPOINT_PATH)
else:
    print("Checkpoint file not found; using the CurricularFace model currently in memory.")

model.eval()

# CurricularFace returns scaled class scores. Remove the scale so the
# threshold is on the model's normalized class-score range.
score_scale = getattr(getattr(model, "head", None), "scale", 1.0)
if isinstance(score_scale, torch.Tensor):
    score_scale = float(score_scale.detach().cpu().item())
else:
    score_scale = float(score_scale)

if score_scale == 0:
    score_scale = 1.0

# ---------- Gather unique known validation images ----------
if not hasattr(validation_data, "samples"):
    raise AttributeError(
        "The validation dataset needs to be an ImageFolder-style dataset with .samples."
    )

known_images = []
known_labels = []
seen_hashes = set()

for image_path, label in validation_data.samples:
    image_path = Path(image_path)

    try:
        digest = hashlib.sha256(image_path.read_bytes()).hexdigest()
        if digest in seen_hashes:
            continue
        seen_hashes.add(digest)

        with Image.open(image_path) as image_file:
            known_images.append(image_file.convert("RGB"))
        known_labels.append(int(label))
    except Exception as error:
        print("Skipped validation image:", image_path.name, error)

if not known_images:
    raise RuntimeError("No unique validation images were found.")

# ---------- Gather one LFW image per unknown identity ----------
LFW_LIMIT = 150
seed = int(globals().get("SEED", 42))
rng = random.Random(seed)

lfw = fetch_lfw_people(
    data_home=LFW_CACHE_DIR,
    min_faces_per_person=2,
    resize=1.0,
    color=True,
    download_if_missing=True
)

team_names = {
    str(name).strip().lower()
    for name in folder_to_name.values()
}

images_by_identity = {}
for image_index, identity_index in enumerate(lfw.target):
    images_by_identity.setdefault(int(identity_index), []).append(image_index)

eligible_identities = [
    identity
    for identity in images_by_identity
    if str(lfw.target_names[identity]).strip().lower() not in team_names
]
rng.shuffle(eligible_identities)

selected_indices = []
for identity in eligible_identities[:LFW_LIMIT]:
    selected_indices.append(rng.choice(images_by_identity[identity]))

unknown_images = []
for image_index in selected_indices:
    image_array = np.asarray(lfw.images[image_index])
    image_array = np.clip(image_array, 0, 255).astype(np.uint8)
    unknown_images.append(Image.fromarray(image_array).convert("RGB"))

if not unknown_images:
    raise RuntimeError("No LFW unknown faces were selected.")

# ---------- Get each image's highest CurricularFace class score ----------
@torch.no_grad()
def get_max_class_scores(images, batch_size=16):
    all_scores = []

    for start in range(0, len(images), batch_size):
        image_batch = images[start:start + batch_size]
        tensors = torch.stack([
            evaluation_transform(image)
            for image in image_batch
        ]).to(device)

        outputs = model(tensors)
        scores = outputs / score_scale
        all_scores.extend(scores.max(dim=1).values.detach().cpu().tolist())

    return all_scores

known_scores = get_max_class_scores(known_images)
unknown_scores = get_max_class_scores(unknown_images)

# ---------- Select a threshold ----------
# Keep at least 95% of the unique known validation faces accepted,
# then choose the threshold that rejects the most LFW unknown faces.
MIN_KNOWN_ACCEPTANCE = 0.95
candidate_thresholds = sorted(set(known_scores + unknown_scores))

valid_choices = []
for threshold in candidate_thresholds:
    known_acceptance = sum(
        score >= threshold for score in known_scores
    ) / len(known_scores)

    unknown_rejection = sum(
        score < threshold for score in unknown_scores
    ) / len(unknown_scores)

    if known_acceptance >= MIN_KNOWN_ACCEPTANCE:
        valid_choices.append(
            (threshold, known_acceptance, unknown_rejection)
        )

if not valid_choices:
    raise RuntimeError(
        "Could not find a threshold that accepts at least 95% of known "
        "validation faces. Check the dataset and model scores."
    )

threshold, known_acceptance, unknown_rejection = max(
    valid_choices,
    key=lambda result: (result[2], result[1], result[0])
)

# Keep the "threshold" key so the existing webcam cell can still load it.
settings = {
    "architecture": "CurricularFace",
    "threshold": float(threshold),
    "unknown_rejection_threshold": float(threshold),
    "class_names": [
        idx_to_name[index]
        for index in sorted(idx_to_name)
    ],
    "id_to_name": {
        str(index): name
        for index, name in idx_to_name.items()
    },
    "unique_known_validation_faces": len(known_scores),
    "lfw_unknown_identities": len(unknown_scores),
    "known_validation_acceptance": float(known_acceptance),
    "lfw_unknown_rejection": float(unknown_rejection),
}

with SETTINGS_PATH.open("w", encoding="utf-8") as settings_file:
    json.dump(settings, settings_file, indent=2)

print("Architecture: CurricularFace")
print("Unique known validation faces:", len(known_scores))
print("Distinct LFW unknown identities:", len(unknown_scores))
print(f"CurricularFace unknown-rejection threshold: {threshold:.4f}")
print(f"Known validation faces accepted: {known_acceptance:.1%}")
print(f"LFW unknown faces rejected: {unknown_rejection:.1%}")
print("Saved settings:", SETTINGS_PATH)

In [ ]:
from pathlib import Path
import json
import cv2
import torch
from PIL import Image
from facenet_pytorch import MTCNN

# Load the saved CurricularFace rejection threshold and member names.
RESULTS_DIR = Path(
    r"curricularface_results"
)
SETTINGS_PATH = RESULTS_DIR / "curricularface_settings.json"
CHECKPOINT_PATH = RESULTS_DIR / "curricularface_best.pt"

with SETTINGS_PATH.open("r", encoding="utf-8") as f:
    settings = json.load(f)

UNKNOWN_THRESHOLD = float(
    settings.get("unknown_rejection_threshold", settings["threshold"])
)
ID_TO_NAME = {
    int(key): value
    for key, value in settings["id_to_name"].items()
}

# Load the best CurricularFace checkpoint.
DEVICE = next(model.parameters()).device

if CHECKPOINT_PATH.is_file():
    try:
        checkpoint = torch.load(
            CHECKPOINT_PATH,
            map_location=DEVICE,
            weights_only=True
        )
    except TypeError:
        checkpoint = torch.load(
            CHECKPOINT_PATH,
            map_location=DEVICE
        )

    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
    else:
        model.load_state_dict(checkpoint)

model.eval()

score_scale = getattr(getattr(model, "head", None), "scale", 1.0)
if isinstance(score_scale, torch.Tensor):
    score_scale = float(score_scale.detach().cpu().item())
else:
    score_scale = float(score_scale)

if score_scale == 0:
    score_scale = 1.0

# Detect and crop faces to the CurricularFace model's 112 x 112 input size.
detector = MTCNN(
    image_size=112,
    margin=14,
    keep_all=True,
    select_largest=False,
    post_process=True,
    device=DEVICE
)

CAMERA_INDEX = 0
MIN_DETECTION_PROBABILITY = 0.90
MIN_FACE_WIDTH_PIXELS = 100


def run_curricularface_webcam(camera_index=CAMERA_INDEX):
    camera = cv2.VideoCapture(camera_index, cv2.CAP_DSHOW)

    if not camera.isOpened():
        camera.release()
        camera = cv2.VideoCapture(camera_index)

    if not camera.isOpened():
        raise RuntimeError(
            "Webcam did not open. Close other camera apps or try CAMERA_INDEX = 1."
        )

    # Ask the webcam for a predictable frame size.
    camera.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    camera.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

    # Create a normal resizable window.
    # Use its title-bar buttons to minimize, maximize, or restore it.
    window_name = "CurricularFace group recognition"
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(window_name, 960, 720)

    print(f"CurricularFace threshold: {UNKNOWN_THRESHOLD:.4f}")
    print("Press Q in the video window to quit.")

    try:
        while True:
            ok, frame = camera.read()
            if not ok:
                print("Could not read a webcam frame.")
                break

            frame_height, frame_width = frame.shape[:2]
            min_face_width = max(
                MIN_FACE_WIDTH_PIXELS,
                int(frame_width * 0.16)
            )
            min_face_height = max(90, int(frame_height * 0.18))

            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_frame = Image.fromarray(rgb_frame)

            boxes, probabilities = detector.detect(pil_frame)
            boxes_to_recognize = []
            boxes_too_small = []

            if boxes is not None and probabilities is not None:
                for box, probability in zip(boxes, probabilities):
                    if probability is None:
                        continue
                    if float(probability) < MIN_DETECTION_PROBABILITY:
                        continue

                    x1, y1, x2, y2 = [int(value) for value in box]
                    face_width = x2 - x1
                    face_height = y2 - y1

                    if (
                        face_width < min_face_width
                        or face_height < min_face_height
                    ):
                        boxes_too_small.append((x1, y1, x2, y2))
                    else:
                        boxes_to_recognize.append(
                            (box, (x1, y1, x2, y2))
                        )

            # A distant/small face is rejected before classification.
            for x1, y1, x2, y2 in boxes_too_small:
                x1 = max(0, x1)
                y1 = max(0, y1)
                x2 = min(frame_width - 1, x2)
                y2 = min(frame_height - 1, y2)

                cv2.rectangle(
                    frame,
                    (x1, y1),
                    (x2, y2),
                    (0, 0, 255),
                    2
                )
                cv2.putText(
                    frame,
                    "unidentified",
                    (x1, max(25, y1 - 10)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.65,
                    (0, 0, 255),
                    2
                )

            # Classify only faces large enough for a useful crop.
            if boxes_to_recognize:
                selected_boxes = [
                    item[0] for item in boxes_to_recognize
                ]
                face_crops = detector.extract(
                    pil_frame,
                    selected_boxes,
                    save_path=None
                )

                if face_crops is not None:
                    if face_crops.ndim == 3:
                        face_crops = face_crops.unsqueeze(0)

                    with torch.no_grad():
                        scores = model(face_crops.to(DEVICE)) / score_scale
                        best_scores, predicted_ids = scores.max(dim=1)

                    for item, score_tensor, id_tensor in zip(
                        boxes_to_recognize,
                        best_scores,
                        predicted_ids
                    ):
                        x1, y1, x2, y2 = item[1]
                        x1 = max(0, x1)
                        y1 = max(0, y1)
                        x2 = min(frame_width - 1, x2)
                        y2 = min(frame_height - 1, y2)

                        score = float(score_tensor.item())
                        predicted_id = int(id_tensor.item())

                        if score < UNKNOWN_THRESHOLD:
                            label = "unidentified"
                            color = (0, 0, 255)
                        else:
                            label = ID_TO_NAME.get(
                                predicted_id,
                                f"member {predicted_id}"
                            )
                            color = (0, 220, 0)

                        cv2.rectangle(
                            frame,
                            (x1, y1),
                            (x2, y2),
                            color,
                            2
                        )
                        cv2.putText(
                            frame,
                            f"{label} ({score:.2f})",
                            (x1, max(25, y1 - 10)),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.65,
                            color,
                            2
                        )

            cv2.putText(
                frame,
                "CurricularFace | Press Q to quit",
                (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.65,
                (0, 255, 255),
                2
            )

            # Display the camera frame in the resizable window.
            cv2.imshow(window_name, frame)

            if cv2.waitKey(1) & 0xFF == ord("q"):
                break

    finally:
        camera.release()
        cv2.destroyAllWindows()


run_curricularface_webcam()